In [10]:
print("fahim")

fahim


In [11]:
# =========================
# AMDNet23 (train/valid folders) Fusion Training Code (PyTorch)
# Directory:
#   DATA_ROOT/
#     train/amd, cataract, diabetes, normal
#     valid/amd, cataract, diabetes, normal
# =========================

import os
import random
import numpy as np
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression


# -------------------------
# Config
# -------------------------
@dataclass
class CFG:
    # CHANGE THIS:
    # Example:
    # ".../AMDNet23 Dataset"  (the folder that has train/ and valid/)
    DATA_ROOT: str = "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset"

    IMG_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2

    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # training
    EPOCHS_HEAD: int = 5         # train classifier head
    EPOCHS_FINE: int = 15        # fine-tune backbone
    LR_HEAD: float = 3e-4
    LR_FINE: float = 1e-5
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05

    # early stopping
    PATIENCE: int = 5

    # fusion condition
    TARGET_ACC: float = 0.98

cfg = CFG()


# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)


# -------------------------
# Paths
# -------------------------
data_root = Path(cfg.DATA_ROOT)
train_dir = data_root / "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset/train"
val_dir   = data_root / "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset/valid"

assert train_dir.exists(), f"Train folder not found: {train_dir}"
assert val_dir.exists(), f"Valid folder not found: {val_dir}"


# -------------------------
# Transforms
# -------------------------
train_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


# -------------------------
# Datasets / Loaders
# -------------------------
train_ds = datasets.ImageFolder(str(train_dir), transform=train_tfms)
val_ds   = datasets.ImageFolder(str(val_dir), transform=val_tfms)

class_names = train_ds.classes
num_classes = len(class_names)

# Sanity checks
assert num_classes == 4, f"Expected 4 classes, got {num_classes}: {class_names}"
assert train_ds.class_to_idx == val_ds.class_to_idx, (
    "Train/Valid class mapping mismatch. Ensure folder names match exactly."
)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)

print("Classes:", class_names)
print("Train size:", len(train_ds), "| Valid size:", len(val_ds))


# -------------------------
# Models to fuse
# -------------------------
MODEL_SPECS = [
    ("tf_efficientnetv2_s", "EffNetV2-S"),
    ("densenet121", "DenseNet121"),
    ("swin_tiny_patch4_window7_224", "Swin-Tiny"),
]


def make_model(model_name: str, num_classes: int):
    return timm.create_model(model_name, pretrained=True, num_classes=num_classes)


def set_trainable_backbone(model, train_backbone: bool):
    # freeze/unfreeze all
    for p in model.parameters():
        p.requires_grad = train_backbone

    # always train classifier head
    for name, p in model.named_parameters():
        if any(k in name.lower() for k in ["classifier", "head", "fc"]):
            p.requires_grad = True


@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    all_probs, all_y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        all_probs.append(probs)
        all_y.append(y.numpy())
    return np.concatenate(all_probs, axis=0), np.concatenate(all_y, axis=0)


@torch.no_grad()
def eval_model(model, loader, device):
    probs, y = predict_proba(model, loader, device)
    pred = probs.argmax(axis=1)
    acc = accuracy_score(y, pred)
    f1m = f1_score(y, pred, average="macro")
    return acc, f1m, probs, y


def train_one_phase(model, train_loader, val_loader, device, epochs, lr, weight_decay,
                    label_smoothing=0.0, patience=5):

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    scaler = torch.cuda.amp.GradScaler(enabled=device.startswith("cuda"))

    best_acc = -1.0
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=device.startswith("cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)

        scheduler.step()

        val_acc, val_f1, _, _ = eval_model(model, val_loader, device)
        tr_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {ep:02d}/{epochs} | loss {tr_loss:.4f} | val_acc {val_acc:.4f} | val_f1 {val_f1:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc


# -------------------------
# Train all base models
# -------------------------
trained_models = []
val_scores = []  # (name, acc, f1)

for timm_name, nice_name in MODEL_SPECS:
    print("\n==============================")
    print("Training:", nice_name, f"({timm_name})")
    print("==============================")

    model = make_model(timm_name, num_classes=num_classes).to(cfg.DEVICE)

    # Phase 1: head
    set_trainable_backbone(model, train_backbone=False)
    model, _ = train_one_phase(
        model, train_loader, val_loader, cfg.DEVICE,
        epochs=cfg.EPOCHS_HEAD,
        lr=cfg.LR_HEAD,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing=cfg.LABEL_SMOOTHING,
        patience=max(2, cfg.PATIENCE // 2),
    )

    # Phase 2: fine-tune
    set_trainable_backbone(model, train_backbone=True)
    model, _ = train_one_phase(
        model, train_loader, val_loader, cfg.DEVICE,
        epochs=cfg.EPOCHS_FINE,
        lr=cfg.LR_FINE,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing=cfg.LABEL_SMOOTHING,
        patience=cfg.PATIENCE,
    )

    acc, f1m, _, _ = eval_model(model, val_loader, cfg.DEVICE)
    print(f"[{nice_name}] Final Val Acc: {acc:.4f} | Val Macro-F1: {f1m:.4f}")

    trained_models.append((nice_name, model))
    val_scores.append((nice_name, acc, f1m))


# -------------------------
# Fusion (condition-based)
# -------------------------
best_single = max(val_scores, key=lambda x: x[1])
best_single_name, best_single_acc, best_single_f1 = best_single
print("\nBest single on VAL:", best_single)

# Collect validation probs
val_probs = {}
val_y = None
for name, model in trained_models:
    probs, y = predict_proba(model, val_loader, cfg.DEVICE)
    val_probs[name] = probs
    val_y = y

def score_probs(probs, y_true):
    pred = probs.argmax(axis=1)
    return accuracy_score(y_true, pred), f1_score(y_true, pred, average="macro")

chosen_strategy = None
chosen_meta = None

if best_single_acc >= cfg.TARGET_ACC:
    chosen_strategy = f"SingleModel:{best_single_name}"
    print(f"\n✅ Condition met: {best_single_name} reached {best_single_acc:.4f} on VAL. Using single model.")
else:
    # Weighted soft-vote using (acc+f1)/2 as weight
    weights = {}
    for n, a, f in val_scores:
        weights[n] = max(1e-6, (a + f) / 2.0)
    s = sum(weights.values())
    for k in weights:
        weights[k] /= s

    soft_vote = np.zeros_like(next(iter(val_probs.values())))
    for n in val_probs:
        soft_vote += weights[n] * val_probs[n]
    soft_acc, soft_f1 = score_probs(soft_vote, val_y)
    print("\nWeighted soft-vote on VAL:", soft_acc, soft_f1, "weights:", weights)

    # Stacking (LogReg) on concatenated probs
    X_val = np.concatenate([val_probs[n] for n, _ in trained_models], axis=1)
    meta = LogisticRegression(max_iter=2000, n_jobs=-1, multi_class="auto")
    meta.fit(X_val, val_y)
    stack_val_probs = meta.predict_proba(X_val)
    stack_acc, stack_f1 = score_probs(stack_val_probs, val_y)
    print("Stacking on VAL:", stack_acc, stack_f1)

    if stack_acc > soft_acc:
        chosen_strategy = "Stacking(LogReg)"
        chosen_meta = meta
    else:
        chosen_strategy = "WeightedSoftVote"

print("\nChosen strategy:", chosen_strategy)

# -------------------------
# Final VAL report (since you provided only train/valid)
# -------------------------
# (If you want a true final test score, you must supply a separate test folder
#  or we can split train into train/test while keeping valid as validation.)
if chosen_strategy.startswith("SingleModel:"):
    chosen_name = chosen_strategy.split(":", 1)[1]
    final_probs = val_probs[chosen_name]
elif chosen_strategy == "WeightedSoftVote":
    final_probs = np.zeros_like(next(iter(val_probs.values())))
    for n in val_probs:
        final_probs += weights[n] * val_probs[n]
elif chosen_strategy == "Stacking(LogReg)":
    X_val = np.concatenate([val_probs[n] for n, _ in trained_models], axis=1)
    final_probs = chosen_meta.predict_proba(X_val)
else:
    raise RuntimeError("Unknown strategy.")

val_pred = final_probs.argmax(axis=1)
val_acc = accuracy_score(val_y, val_pred)
val_f1m = f1_score(val_y, val_pred, average="macro")
cm = confusion_matrix(val_y, val_pred)

print("\n==============================")
print("VALIDATION RESULTS")
print("==============================")
print("Strategy:", chosen_strategy)
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val Macro-F1: {val_f1m:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(val_y, val_pred, target_names=class_names))


Classes: ['amd', 'cataract', 'diabetes', 'normal']
Train size: 1594 | Valid size: 400

Training: EffNetV2-S (tf_efficientnetv2_s)


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

/tmp/ipykernel_47/2417562914.py:183: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.startswith("cuda"))
/tmp/ipykernel_47/2417562914.py:198: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.startswith("cuda")):


Epoch 01/5 | loss 3.4826 | val_acc 0.6925 | val_f1 0.6800


/tmp/ipykernel_47/2417562914.py:198: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.startswith("cuda")):


...

Epoch 04/5 | loss 1.7139 | val_acc 0.7525 | val_f1 0.7523


/tmp/ipykernel_47/2417562914.py:198: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.startswith("cuda")):


Epoch 05/5 | loss 1.5872 | val_acc 0.7500 | val_f1 0.7488


/tmp/ipykernel_47/2417562914.py:183: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.startswith("cuda"))
/tmp/ipykernel_47/2417562914.py:198: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.startswith("cuda")):


KeyboardInterrupt: 

In [12]:
# =========================
# AMDNet23 (train/valid folders) Fusion Training Code (PyTorch)
# Updated AMP API (no deprecation warnings)
#
# Directory:
#   DATA_ROOT/
#     train/amd, cataract, diabetes, normal
#     valid/amd, cataract, diabetes, normal
# =========================

import os
import random
import numpy as np
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression

# ✅ NEW AMP IMPORTS
from torch.amp import autocast, GradScaler


# -------------------------
# Config
# -------------------------
@dataclass
class CFG:
    # CHANGE THIS:
    # Example:
    # ".../AMDNet23 Dataset"  (the folder that has train/ and valid/)
    DATA_ROOT: str = "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset"

    IMG_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2

    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # training
    EPOCHS_HEAD: int = 5         # train classifier head
    EPOCHS_FINE: int = 15        # fine-tune backbone
    LR_HEAD: float = 3e-4
    LR_FINE: float = 1e-5
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05

    # early stopping
    PATIENCE: int = 5

    # fusion condition
    TARGET_ACC: float = 0.98

cfg = CFG()


# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)


# -------------------------
# Paths
# -------------------------
data_root = Path(cfg.DATA_ROOT)
train_dir = data_root / "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset/train"
val_dir   = data_root / "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset/valid"

assert train_dir.exists(), f"Train folder not found: {train_dir}"
assert val_dir.exists(), f"Valid folder not found: {val_dir}"


# -------------------------
# Transforms
# -------------------------
train_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


# -------------------------
# Datasets / Loaders
# -------------------------
train_ds = datasets.ImageFolder(str(train_dir), transform=train_tfms)
val_ds   = datasets.ImageFolder(str(val_dir), transform=val_tfms)

class_names = train_ds.classes
num_classes = len(class_names)

assert num_classes == 4, f"Expected 4 classes, got {num_classes}: {class_names}"
assert train_ds.class_to_idx == val_ds.class_to_idx, (
    "Train/Valid class mapping mismatch. Ensure folder names match exactly."
)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)

print("Classes:", class_names)
print("Train size:", len(train_ds), "| Valid size:", len(val_ds))


# -------------------------
# Models to fuse
# -------------------------
MODEL_SPECS = [
    ("tf_efficientnetv2_s", "EffNetV2-S"),
    ("densenet121", "DenseNet121"),
    ("swin_tiny_patch4_window7_224", "Swin-Tiny"),
]


def make_model(model_name: str, num_classes: int):
    return timm.create_model(model_name, pretrained=True, num_classes=num_classes)


def set_trainable_backbone(model, train_backbone: bool):
    for p in model.parameters():
        p.requires_grad = train_backbone

    # always train classifier head
    for name, p in model.named_parameters():
        if any(k in name.lower() for k in ["classifier", "head", "fc"]):
            p.requires_grad = True


@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    all_probs, all_y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        all_probs.append(probs)
        all_y.append(y.numpy())
    return np.concatenate(all_probs, axis=0), np.concatenate(all_y, axis=0)


@torch.no_grad()
def eval_model(model, loader, device):
    probs, y = predict_proba(model, loader, device)
    pred = probs.argmax(axis=1)
    acc = accuracy_score(y, pred)
    f1m = f1_score(y, pred, average="macro")
    return acc, f1m, probs, y


def train_one_phase(model, train_loader, val_loader, device, epochs, lr, weight_decay,
                    label_smoothing=0.0, patience=5):

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    # ✅ UPDATED: torch.amp.GradScaler
    scaler = GradScaler("cuda", enabled=device.startswith("cuda"))

    best_acc = -1.0
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # ✅ UPDATED: torch.amp.autocast
            with autocast("cuda", enabled=device.startswith("cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)

        scheduler.step()

        val_acc, val_f1, _, _ = eval_model(model, val_loader, device)
        tr_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {ep:02d}/{epochs} | loss {tr_loss:.4f} | val_acc {val_acc:.4f} | val_f1 {val_f1:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc


# -------------------------
# Train all base models
# -------------------------
trained_models = []
val_scores = []  # (name, acc, f1)

for timm_name, nice_name in MODEL_SPECS:
    print("\n==============================")
    print("Training:", nice_name, f"({timm_name})")
    print("==============================")

    model = make_model(timm_name, num_classes=num_classes).to(cfg.DEVICE)

    # Phase 1: head
    set_trainable_backbone(model, train_backbone=False)
    model, _ = train_one_phase(
        model, train_loader, val_loader, cfg.DEVICE,
        epochs=cfg.EPOCHS_HEAD,
        lr=cfg.LR_HEAD,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing=cfg.LABEL_SMOOTHING,
        patience=max(2, cfg.PATIENCE // 2),
    )

    # Phase 2: fine-tune
    set_trainable_backbone(model, train_backbone=True)
    model, _ = train_one_phase(
        model, train_loader, val_loader, cfg.DEVICE,
        epochs=cfg.EPOCHS_FINE,
        lr=cfg.LR_FINE,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing=cfg.LABEL_SMOOTHING,
        patience=cfg.PATIENCE,
    )

    acc, f1m, _, _ = eval_model(model, val_loader, cfg.DEVICE)
    print(f"[{nice_name}] Final Val Acc: {acc:.4f} | Val Macro-F1: {f1m:.4f}")

    trained_models.append((nice_name, model))
    val_scores.append((nice_name, acc, f1m))


# -------------------------
# Fusion (condition-based)
# -------------------------
best_single = max(val_scores, key=lambda x: x[1])
best_single_name, best_single_acc, best_single_f1 = best_single
print("\nBest single on VAL:", best_single)

val_probs = {}
val_y = None
for name, model in trained_models:
    probs, y = predict_proba(model, val_loader, cfg.DEVICE)
    val_probs[name] = probs
    val_y = y

def score_probs(probs, y_true):
    pred = probs.argmax(axis=1)
    return accuracy_score(y_true, pred), f1_score(y_true, pred, average="macro")

chosen_strategy = None
chosen_meta = None

if best_single_acc >= cfg.TARGET_ACC:
    chosen_strategy = f"SingleModel:{best_single_name}"
    print(f"\n✅ Condition met: {best_single_name} reached {best_single_acc:.4f} on VAL. Using single model.")
else:
    # Weighted soft-vote using (acc+f1)/2 as weight
    weights = {}
    for n, a, f in val_scores:
        weights[n] = max(1e-6, (a + f) / 2.0)
    s = sum(weights.values())
    for k in weights:
        weights[k] /= s

    soft_vote = np.zeros_like(next(iter(val_probs.values())))
    for n in val_probs:
        soft_vote += weights[n] * val_probs[n]
    soft_acc, soft_f1 = score_probs(soft_vote, val_y)
    print("\nWeighted soft-vote on VAL:", soft_acc, soft_f1, "weights:", weights)

    # Stacking (LogReg) on concatenated probs
    X_val = np.concatenate([val_probs[n] for n, _ in trained_models], axis=1)
    meta = LogisticRegression(max_iter=2000, n_jobs=-1, multi_class="auto")
    meta.fit(X_val, val_y)
    stack_val_probs = meta.predict_proba(X_val)
    stack_acc, stack_f1 = score_probs(stack_val_probs, val_y)
    print("Stacking on VAL:", stack_acc, stack_f1)

    if stack_acc > soft_acc:
        chosen_strategy = "Stacking(LogReg)"
        chosen_meta = meta
    else:
        chosen_strategy = "WeightedSoftVote"

print("\nChosen strategy:", chosen_strategy)

# -------------------------
# Final VAL report
# -------------------------
if chosen_strategy.startswith("SingleModel:"):
    chosen_name = chosen_strategy.split(":", 1)[1]
    final_probs = val_probs[chosen_name]
elif chosen_strategy == "WeightedSoftVote":
    final_probs = np.zeros_like(next(iter(val_probs.values())))
    for n in val_probs:
        final_probs += weights[n] * val_probs[n]
elif chosen_strategy == "Stacking(LogReg)":
    X_val = np.concatenate([val_probs[n] for n, _ in trained_models], axis=1)
    final_probs = chosen_meta.predict_proba(X_val)
else:
    raise RuntimeError("Unknown strategy.")

val_pred = final_probs.argmax(axis=1)
val_acc = accuracy_score(val_y, val_pred)
val_f1m = f1_score(val_y, val_pred, average="macro")
cm = confusion_matrix(val_y, val_pred)

print("\n==============================")
print("VALIDATION RESULTS")
print("==============================")
print("Strategy:", chosen_strategy)
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val Macro-F1: {val_f1m:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(val_y, val_pred, target_names=class_names))


Classes: ['amd', 'cataract', 'diabetes', 'normal']
Train size: 1594 | Valid size: 400

Training: EffNetV2-S (tf_efficientnetv2_s)
Epoch 01/5 | loss 3.4826 | val_acc 0.6925 | val_f1 0.6800
Epoch 02/5 | loss 2.1614 | val_acc 0.7000 | val_f1 0.6904
Epoch 03/5 | loss 1.7006 | val_acc 0.7400 | val_f1 0.7372
Epoch 04/5 | loss 1.7139 | val_acc 0.7525 | val_f1 0.7523
Epoch 05/5 | loss 1.5872 | val_acc 0.7500 | val_f1 0.7488
Epoch 01/15 | loss 1.6053 | val_acc 0.7675 | val_f1 0.7654
Epoch 02/15 | loss 1.4643 | val_acc 0.7725 | val_f1 0.7722
Epoch 03/15 | loss 1.3270 | val_acc 0.7800 | val_f1 0.7809
Epoch 04/15 | loss 1.3482 | val_acc 0.8000 | val_f1 0.7991
Epoch 05/15 | loss 1.1913 | val_acc 0.8025 | val_f1 0.8023
Epoch 06/15 | loss 1.0976 | val_acc 0.8000 | val_f1 0.7995
Epoch 07/15 | loss 1.1173 | val_acc 0.8100 | val_f1 0.8092
Epoch 08/15 | loss 1.0020 | val_acc 0.8200 | val_f1 0.8191
Epoch 09/15 | loss 1.0287 | val_acc 0.8200 | val_f1 0.8185
Epoch 10/15 | loss 0.9798 | val_acc 0.8325 | val_

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Epoch 01/5 | loss 1.1752 | val_acc 0.6975 | val_f1 0.6827
Epoch 02/5 | loss 0.8910 | val_acc 0.7750 | val_f1 0.7726
Epoch 03/5 | loss 0.7860 | val_acc 0.7825 | val_f1 0.7747
Epoch 04/5 | loss 0.7583 | val_acc 0.7975 | val_f1 0.7948
Epoch 05/5 | loss 0.7360 | val_acc 0.8000 | val_f1 0.7977
Epoch 01/15 | loss 0.7207 | val_acc 0.8150 | val_f1 0.8123
Epoch 02/15 | loss 0.6704 | val_acc 0.8350 | val_f1 0.8319
Epoch 03/15 | loss 0.6219 | val_acc 0.8450 | val_f1 0.8418
Epoch 04/15 | loss 0.5945 | val_acc 0.8500 | val_f1 0.8460
Epoch 05/15 | loss 0.5684 | val_acc 0.8625 | val_f1 0.8613
Epoch 06/15 | loss 0.5545 | val_acc 0.8725 | val_f1 0.8700
Epoch 07/15 | loss 0.5360 | val_acc 0.8750 | val_f1 0.8738
Epoch 08/15 | loss 0.5204 | val_acc 0.8850 | val_f1 0.8831
Epoch 09/15 | loss 0.5084 | val_acc 0.8750 | val_f1 0.8739
Epoch 10/15 | loss 0.5017 | val_acc 0.8900 | val_f1 0.8882
Epoch 11/15 | loss 0.4885 | val_acc 0.8850 | val_f1 0.8837
Epoch 12/15 | loss 0.4983 | val_acc 0.8925 | val_f1 0.8906
Ep

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 01/5 | loss 0.6825 | val_acc 0.8950 | val_f1 0.8942
Epoch 02/5 | loss 0.4070 | val_acc 0.9350 | val_f1 0.9347
Epoch 03/5 | loss 0.3230 | val_acc 0.9625 | val_f1 0.9624
Epoch 04/5 | loss 0.2679 | val_acc 0.9800 | val_f1 0.9799
Epoch 05/5 | loss 0.2543 | val_acc 0.9725 | val_f1 0.9722
Epoch 01/15 | loss 0.2532 | val_acc 0.9700 | val_f1 0.9699
Epoch 02/15 | loss 0.2497 | val_acc 0.9725 | val_f1 0.9723
Epoch 03/15 | loss 0.2355 | val_acc 0.9725 | val_f1 0.9723
Epoch 04/15 | loss 0.2376 | val_acc 0.9725 | val_f1 0.9723
Epoch 05/15 | loss 0.2303 | val_acc 0.9725 | val_f1 0.9723
Epoch 06/15 | loss 0.2333 | val_acc 0.9700 | val_f1 0.9696
Epoch 07/15 | loss 0.2269 | val_acc 0.9725 | val_f1 0.9723
Early stopping.
[Swin-Tiny] Final Val Acc: 0.9725 | Val Macro-F1: 0.9723

Best single on VAL: ('Swin-Tiny', 0.9725, 0.9723186653620499)

Weighted soft-vote on VAL: 0.96 0.9594532626871135 weights: {'EffNetV2-S': 0.30847137175074035, 'DenseNet121': 0.3322178519049439, 'Swin-Tiny': 0.35931077634431